# Embeddings and Optional Pinecone Sync

Notebook ini membuat embeddings untuk seluruh dataset prepared hasil parsing raw CSV, lalu opsional melakukan sinkronisasi ke Pinecone.


In [1]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import joblib
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

from src.config import ARTIFACTS_DIR, INTERIM_DIR, load_environment
from src.preprocessing import load_prepared_dataset
from src.pinecone_utils import build_client, create_index_if_missing, upsert_embeddings

MODEL_NAME = 'all-MiniLM-L6-v2'


c:\porto\NemuParfang\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def safe_int(value, default=0):
    if pd.isna(value) or value is None or value == '':
        return default
    return int(float(value))

def safe_float(value, default=0.0):
    if pd.isna(value) or value is None or value == '':
        return default
    return float(value)

def safe_str(value, default=''):
    if pd.isna(value) or value is None:
        return default
    text = str(value).strip()
    if not text or text.lower() in {'nan', 'none', '<na>', '0'}:
        return default
    return text

settings = load_environment()
df = load_prepared_dataset(INTERIM_DIR)
encoder = SentenceTransformer(MODEL_NAME)
embedding_texts = df['perfume_text'].fillna('').tolist()
embeddings = encoder.encode(embedding_texts, batch_size=128, show_progress_bar=True)
embeddings = np.asarray(embeddings, dtype='float32')

joblib.dump(embeddings, ARTIFACTS_DIR / 'perfume_embeddings.joblib')
with (ARTIFACTS_DIR / 'embedding_config.json').open('w', encoding='utf-8') as handle:
    json.dump({'model_name': MODEL_NAME, 'embedding_dim': int(embeddings.shape[1]), 'row_count': int(len(df))}, handle, indent=2)
print('Loaded rows:', len(df))
print('Embeddings shape:', embeddings.shape)


Batches: 100%|██████████| 548/548 [06:28<00:00,  1.41it/s]


Loaded rows: 70103
Embeddings shape: (70103, 384)


In [3]:
if settings.api_key:
    client = build_client(settings.api_key)
    create_index_if_missing(client, settings.index_name, embeddings.shape[1], settings.cloud, settings.region)
    index = client.Index(settings.index_name)

    rows = []
    for i, row in tqdm(df.iterrows(), total=len(df)):
        rows.append({
            'id': str(row['id']),
            'values': embeddings[i].tolist(),
            'metadata': {
                'name': safe_str(row.get('perfume_name', '')),
                'brand': safe_str(row.get('brand', '')),
                'country': safe_str(row.get('country', '')),
                'gender': safe_str(row.get('gender', '')),
                'year': safe_int(row.get('year', 0)),
                'rating': safe_float(row.get('rating', 0)),
                'review_count': safe_int(row.get('review_count', 0)),
                'accords': safe_str(row.get('accords', '')),
            },
        })

    upsert_embeddings(index, rows, batch_size=100)
    print('Synced embeddings to Pinecone.')
else:
    print('Pinecone key not set, skipping sync. Embeddings are still saved locally.')


100%|██████████| 70103/70103 [00:03<00:00, 21421.68it/s]


Synced embeddings to Pinecone.


In [4]:
query = 'warm woody amber vanilla date night'
query_vec = encoder.encode(query)
print('Query embedding ready:', len(query_vec))


Query embedding ready: 384
